In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import gradio as gr

# Load model 1 - DistilBERT
model1_path = "models/saved_model_distilbert"
tokenizer1 = AutoTokenizer.from_pretrained(model1_path)
model1 = AutoModelForSequenceClassification.from_pretrained(model1_path)
model1.eval()

# Load model 2 - Electra
model2_path = "models/saved_model_electra"
tokenizer2 = AutoTokenizer.from_pretrained(model2_path)
model2 = AutoModelForSequenceClassification.from_pretrained(model2_path)
model2.eval()

# Load model 3 - MiniLM
model3_path = "models/nreimers MiniLM Model"
tokenizer3 = AutoTokenizer.from_pretrained(model3_path)
model3 = AutoModelForSequenceClassification.from_pretrained(model3_path)
model3.eval()

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model1.to(device)
model2.to(device)
model3.to(device)

# Shared classification function
def classify_text(text):
    def predict(model, tokenizer):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        return probs.cpu().numpy().tolist()[0]

    result1 = predict(model1, tokenizer1)
    result2 = predict(model2, tokenizer2)
    result3 = predict(model3, tokenizer3)

    labels = ["Not Depressed", "Depressed"]

    def format_prediction(result):
        pred_index = int(result[1] > result[0])  # 1 if Depressed > Not Depressed
        pred_label = labels[pred_index]
        confidence = result[pred_index]
        return f"{pred_label} ({confidence * 100:.2f}%)"

    return (
        format_prediction(result1),
        format_prediction(result2),
        format_prediction(result3)
    )

# Gradio interface
iface = gr.Interface(
    fn=classify_text,
    inputs=gr.Textbox(label="Enter a sentence"),
    outputs=[
        gr.Label(label="DistilBERT Prediction"),
        gr.Label(label="Electra Prediction"),
        gr.Label(label="MiniLM Prediction")
    ],
    title="Depression Detection using 3 Models",
    description="This tool uses 3 different transformer models to predict if the sentence indicates depression or not."
)

iface.launch()


C:\Users\USER\anaconda3\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\USER\anaconda3\envs\torch\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
C:\Users\USER\anaconda3\envs\torch\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
C:\Users\USER\anaconda3\envs\torch\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
